<a href="https://colab.research.google.com/github/Ivan5245/My_Projects/blob/main/NLP_sent/Untitled47.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import warnings
import math
import torch
from torch import nn
import re
warnings.filterwarnings('ignore', category=DeprecationWarning)
data=pd.read_csv('https://raw.githubusercontent.com/Ivan5245/My_Projects/refs/heads/main/NLP_sent/sentiment_dataset.csv')
print(np.unique(data.label),np.unique(data.sentiment),)
data.head()

[0 1 2] ['negative' 'neutral' 'positive']


,id,text,label,sentiment
0,0,"Cooking microwave pizzas, yummy",2,positive
1,1,Any plans of allowing sub tasks to show up in ...,1,neutral
2,2,"I love the humor, I just reworded it. Like sa...",2,positive
3,3,naw idk what ur talkin about,1,neutral
4,4,That sucks to hear. I hate days like that,0,negative


In [8]:
def f(s):
  s=s.lower()
  s=re.sub(r'[^a-z,!?. ]','',s)
  s=re.sub(r'([,.!?])',' \1',s)
  s=s.split()
  return s
Text=[f(s) for s in data.text.values]
Words=np.unique(sum(Text,[]))

In [ ]:
print(len(Words))

Embadding training

In [ ]:
class MyData(torch.utils.data.Dataset):
  def __init__(self,data):
    super().__init__()
    self.data=data
    self.Text=[f(s) for s in data.text.values]
    self.Words=np.unique(sum(self.Text,[]))
    self.W2I=dict(enumerate(self.Words))
    self.I2W={a:b for b,a in self.W2I.items()}
    self.shape=len(self.Words)
    self.OHE=torch.eye(self.shape)
    self.text=[torch.tensor([self.W2I[w] for w in s]) for s in self.Text]
    self.answ=[[] for i in range(self.shape)]
    for s in self.text:
      for i in range(len(s)):
        for j in range(i+1,len(s)):
          self.answ[s[i]].append([s[j],j-i])
          self.answ[s[j]].append([s[i],j-i])
  def pois(k):
    if(k==0):
      return 0
    return math.exp(-2)*(2**k)/math.factorial(k)
  def smooth(self,s):
    res=torch.zeros(self.shape)
    for i in self.answ[s]:
      res+=self.OHE[i[0]]*self.pois(i[1])
    return res
  def __getitem__(self,id):
    return self.OHE[id],self.smooth(id)

  def __len__(self):
    return self.shape
Dat=MyData(data)
Load=torch.utils.DataLoader(Dat,shuffle=True,batch_size=10)

In [ ]:
class MyModel(nn.Module):
  def __init__(self,num,a):
    super().__init__()
    self.l1=nn.Linear(num,a)
    self.l2=nn.Linear(a,num)
  def forward(self,x):
    x=self.l1(x)
    x=torch.relu(x)
    x=self.l2(x)
model=MyModel(Dat.shape,100)
opt=torch.optim.Adam(model.parameters(),lr=0.01)
loss_func=torch.MSELoss()
for _ in range(100):
  for x,y in Load:
    pred=model(x)
    loss=loss_func(pred,y)
    opt.zero_grad()
    loss.backward()
    opt.step()